# Agent 智能体设计模式

## ReAct
随着 LLM 在关键能力（理解复杂输入、进行推理和规划、可靠地使用工具以及从错误中恢复）方面的日趋成熟，代理正在投入生产。代理可以通过人类用户的命令或与人类用户的互动讨论开始工作。一旦任务明确，代理就会独立规划和操作，并可能返回给人类以获取更多信息或判断。

![](https://i-blog.csdnimg.cn/direct/557f400632b2446e9b47e8e3525b652e.png)

- 开放性问题无法事先定义工作流
- 大模型自行规划执行条件和步骤
- 根据外部反馈进行下一步行动

In [1]:
from openai import OpenAI
from datetime import datetime
import json
from typing import List, Dict, Callable
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
load_dotenv("/Users/a1-6/Documents/projects/DL/.env")
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

llm_name = llm_name = "qwen-plus"
def call_llm(user_prompt, system_prompt=""):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt}
        ]
    response = client.chat.completions.create(
            model= llm_name,
            messages = messages)
    return(response.choices[0].message.content)

In [ ]:
system_prompt = """
You are a ReAct (Reasoning and Acting) agent that follows a loop of Thought, Action, PAUSE, and Observation to solve problems step-by-step.

Workflow I
1. Thought: Describe your reasoning or plan for solving the problem.
2. Action: Execute an appropriate action based on your reasoning. The available actions are listed below.
3. PAUSE: Indicate that you are pausing to observe the result of the action. stop output anything while pause.
4. Observation: Analyze the result of the action and incorporate it into your reasoning.

At the end of the loop, provide a final Answer based on the information gathered.

Your available actions are:

    calculate:
    e.g. calculate: 5*7/4
    run a calculation and returns the number using python so be sure to use floating point syntax if needed.

    planet_mass:
    e.g. planet_mass: Mars
    returns the mass of the planet in the solar system
"""

In [2]:
def calculate(what):
    return eval(what)


def planet_mass(planet):
    masses = {
        "Mercury.": 0.3301,
        "Venus": 4.8557,
        "Earth": 5.972,
        "Mars": 0.6418,
        "Jupiter": 1898.2,
        "Saturn": 568.34,
        "Uranus": 86.82,
        "Neptune": 102.4,
        }
    return f"{planet} has a mass of {masses[planet]} 10^24 kg"

know_actions = {
    "calculate": calculate,
    "planet_mass": planet_mass
}

In [3]:
class Agent:
    def __init__(self,system=None):
        self.system = system
        self.api_key = os.getenv("DASHSCOPE_API_KEY")
        self.base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        self.client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        self.model = "qwen-plus"
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        response = self.client.chat.completions.create(
                   model= self.model,
                   messages = self.messages)
        return response.choices[0].message.content
    


In [ ]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(system_prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print("----------------------")
        print(f"step: {i}")
        print(result)
        action_re = re.compile(r"^Action:\s*(\w+):\s*(.*)$")
        actions = [action_re.match(line) for line in result.split("\n") if action_re.match(line)]
        if actions:
            print("\nlet's do something")
            action, action_input = actions[0].groups()
            if action not in know_actions:
                raise Exception(f"Unknown action: {action}")
            print(f"\nRunning {action}({action_input})")
            observation = know_actions[action](action_input)
            print(f"\nObservation: {observation}")
            next_prompt = f"Observation: {observation}"
        else:
            return

